In [ ]:
from scripts.request_b3_data import request_report
#request_report(date='2026-05-31')

In [ ]:
from scripts.format_data import format_

format_(ref_data='2607')

# extract data from pdf

In [ ]:
import pdfplumber
path="data_lake/market_summary/BDI_02-1_20260630.pdf"

complete_text=[]
with pdfplumber.open(path) as pdf:
    file_pages= pdf.pages

    for page_i in file_pages:
        
        complete_text.append(page_i.extract_text().split('\n'))

In [ ]:
file_table=[[page_id+1,row] for page_id,sub_list in enumerate(complete_text) for row in sub_list ]

from pandas import DataFrame 
file_table=DataFrame(file_table,columns=['page','content'])

In [ ]:
#common rows to remove
remove_rows=['Boletim Diário do Mercado',
             'Resumo de ações']
file_table=file_table[~file_table['content'].isin(remove_rows)]

from scripts.map_market_summary import map_market_summary

#mark tables headers
tables_headers={
'Ações - resumo das operações':{'ticket_summary'},
'Comportamento dos valores de referência das cotas dos ETFs (IOPV)':'tab_teste_1',
'Mercado a termo':'tab_teste_2',
'Negociação de estratégias':'tab_teste_3',
'Negócios consolidados do pregão':'tab_teste_4',
'Negócios consolidados do pregão não regular':'tab_teste_5'
}

filter_tab_header=file_table['content'].isin(tables_headers.keys())
file_table.loc[filter_tab_header,'TABLE_REF']=file_table.loc[filter_tab_header,'content'].map(tables_headers)
file_table['TABLE_REF']=file_table['TABLE_REF'].ffill().fillna('-')

file_table['ROW_ID']=file_table.groupby('TABLE_REF').cumcount()

In [63]:
from scripts.map_market_summary import map_market_summary

 

In [90]:
class map_element():
    def __init__(self,table_name="map_market_summary"):
        self.selected_table=table_name
        self.__define()

    def __define(self):
        from scripts.map_market_summary import map_market_summary
        string_to_ref={'map_market_summary':map_market_summary}
        self.ref_table=string_to_ref[self.selected_table]

        self.__standard_transformation()
    
    def __standard_transformation(self):
        from pandas import DataFrame
        df_element=[]
        for i,j in self.ref_table.items():
            df_element.append([
                    i,"table_id",i
                ])
            for x,y in j.items():

                df_element.append([
                    i,x,y
                ])

        df_element=DataFrame(df_element,
                             columns=['table_id','atribute','value'])
 
 
        self._table_configs=df_element

   

map_table=map_element(table_name='map_market_summary')       

In [92]:
map_table._table_configs

,table_id,atribute,value
0,table_0,table_id,table_0
1,table_0,comment,[sample table]
2,table_0,original_table_name,
3,table_0,new_table_name,0
4,table_0,table_header,[]
5,table_1,table_id,table_1
6,table_1,comment,[]
7,table_1,original_table_name,Ações - resumo das operações
8,table_1,new_table_name,1
9,table_1,table_header,[]


In [78]:
map_table={value: table
    for table,atributes in map_market_summary.items() 
    for key,value in atributes.items() 
    if key=='original_table_name' }
map_table

{'': 'table_0',
 'Ações - resumo das operações': 'table_1',
 'Comportamento dos valores de referência das cotas dos ETFs (IOPV)': 'table_2',
 'Mercado a termo': 'table_3',
 'Negociação de estratégias': 'table_4',
 'Negócios consolidados do pregão': 'table_5',
 'Negócios consolidados do pregão não regular': 'table_6'}

In [ ]:
filter=(
    #(file_table['page']==1)&
    (file_table['ROW_ID'].isin([0,1,2]))
)
file_table[filter]

In [ ]:
file_table[file_table['TABLE_REF']=='tab_teste_3']

In [ ]:
tabelas=[
'Ações - resumo das operações',
'Comportamento dos valores de referência das cotas dos ETFs (IOPV)',
'Mercado a termo',
'Negociação de estratégias',
'Negócios consolidados do pregão',
'Negócios consolidados do pregão não regular'

]

In [ ]:
map_tables={
    'Ações - resumo das operações':[
        'Discriminação Número de negócios Títulos mil Participação (%) Valor (R$) mil Participação (%)',
        ['Discriminação','Número de negócios','Títulos mil','Participação (%)','Valor (R$) mil','Participação (%)']
    ]
}

In [ ]:
text 